# Practice 4

# 📓 Exercise 4 — Complete Standalone Notebook

# Exercise 4: Comprehensive Model Comparison Table (CV RMSE vs. Test RMSE)

**Goal:** Build a consolidated summary table containing all models trained throughout the notebook (Linear Regression, Default Random Forest, Gradient Boosting, GridSearchCV-tuned Random Forest, and RandomizedSearchCV-tuned Random Forest). Compare both their 5-fold Cross-Validation RMSE (on training data) and Test RMSE (on unseen test data) to evaluate predictive performance and generalization capability.

# Cell 1 — Import required libraries and modules

We import all required modules for data loading, preprocessing, model selection, evaluation metrics, and regression models.

In [9]:
# Import core tools, preprocessing, model evaluation, and regressors.

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error

# Regressors

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Cell 2 — Load dataset and engineer features

We load the California Housing dataset, create synthetic missing values, and construct categorical and numerical engineered features.

In [10]:
# Load dataset and prepare reproducible messy features.

california = fetch_california_housing(as_frame=True)
housing = california.frame

rng = np.random.default_rng(42)
messy = housing.copy()

# Introduce missing values

missing_idx = rng.choice(messy.index, size=200, replace=False)
messy.loc[missing_idx, "AveBedrms"] = np.nan
missing_idx_2 = rng.choice(messy.index, size=120, replace=False)
messy.loc[missing_idx_2, "AveRooms"] = np.nan

# Feature engineering

messy["coast_region"] = pd.cut(
    messy["Longitude"],
    bins=[-125, -121.22, -118.20, -114],
    labels=["Coastal", "Central", "Inland"],)

messy["income_cat"] = pd.cut(
    messy["MedInc"],
    bins=[0, 2.5, 4.5, 7, np.inf],
    labels=["Low", "Medium", "High", "Very High"],)

messy["bedrooms_ratio"] = messy["AveBedrms"] / messy["AveRooms"]
messy["rooms_per_person"] = messy["AveRooms"] / messy["AveOccup"]

X = messy.drop(columns="MedHouseVal")
y = messy["MedHouseVal"]

# Cell 3 — Split dataset into training and test sets

We split the features and target variable into an 80% training set and a 20% test set using a fixed random seed.

In [11]:
# Train-test split (80% train, 20% test).

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

numeric_features = [
    "MedInc", "HouseAge", "AveRooms", "AveBedrms",
    "Population", "AveOccup", "Latitude", "Longitude"]

categorical_features = ["coast_region"]

# Cell 4 — Define the preprocessor pipeline

We set up a ColumnTransformer for median imputation and scaling on numeric columns, and most-frequent imputation and one-hot encoding on categorical columns.

In [12]:
# Construct numerical and categorical preprocessing pipelines.

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),])

# Cell 5 — Evaluate Model 1: Linear Regression

We calculate 5-fold CV RMSE on training set, then fit on X_train and evaluate Test RMSE on X_test.

In [13]:
# Model 1: Linear Regression

pipe_lr = Pipeline([("preprocessor", preprocessor), ("model", LinearRegression())])

# 5-fold CV RMSE

cv_lr = -cross_val_score(pipe_lr, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error").mean()

# Test RMSE

pipe_lr.fit(X_train, y_train)
test_lr = np.sqrt(mean_squared_error(y_test, pipe_lr.predict(X_test)))

print(f"Linear Regression -> CV RMSE: {cv_lr:.4f} | Test RMSE: {test_lr:.4f}")

Linear Regression -> CV RMSE: 0.7184 | Test RMSE: 0.7441


# Cell 6 — Evaluate Model 2: Default Random Forest Regressor

We calculate 5-fold CV RMSE and Test RMSE for the un-tuned Random Forest Regressor.

In [14]:
# Model 2: Default Random Forest

pipe_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))])

# 5-fold CV RMSE

cv_rf = -cross_val_score(pipe_rf, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error").mean()

# Test RMSE

pipe_rf.fit(X_train, y_train)
test_rf = np.sqrt(mean_squared_error(y_test, pipe_rf.predict(X_test)))

print(f"Random Forest (Default) -> CV RMSE: {cv_rf:.4f} | Test RMSE: {test_rf:.4f}")

Random Forest (Default) -> CV RMSE: 0.5091 | Test RMSE: 0.5039


# Cell 7 — Evaluate Model 3: Gradient Boosting Regressor

We evaluate Gradient Boosting Regressor (introduced in Exercise 1) using 5-fold CV RMSE and Test RMSE.

In [15]:
# Model 3: Gradient Boosting

pipe_gb = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(n_estimators=100, random_state=42))])

# 5-fold CV RMSE

cv_gb = -cross_val_score(pipe_gb, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error").mean()

# Test RMSE

pipe_gb.fit(X_train, y_train)
test_gb = np.sqrt(mean_squared_error(y_test, pipe_gb.predict(X_test)))

print(f"Gradient Boosting -> CV RMSE: {cv_gb:.4f} | Test RMSE: {test_gb:.4f}")

Gradient Boosting -> CV RMSE: 0.5352 | Test RMSE: 0.5417


# Cell 8 — Evaluate Model 4: Tuned Random Forest via GridSearchCV

We evaluate the best model selected by GridSearchCV in Section 10.

In [16]:
# Model 4: GridSearchCV Tuned Random Forest

param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20],}

pipe_grid = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))])

grid_search = GridSearchCV(pipe_grid, param_grid, cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
grid_search.fit(X_train, y_train)

cv_grid = -grid_search.best_score_
test_grid = np.sqrt(mean_squared_error(y_test, grid_search.best_estimator_.predict(X_test)))

print(f"GridSearchCV RF -> CV RMSE: {cv_grid:.4f} | Test RMSE: {test_grid:.4f}")

GridSearchCV RF -> CV RMSE: 0.5091 | Test RMSE: 0.5039


# Cell 9 — Evaluate Model 5: Tuned Random Forest via RandomizedSearchCV

We evaluate the best model selected by RandomizedSearchCV in Section 11.

In [17]:
# Model 5: RandomizedSearchCV Tuned Random Forest

param_distributions = {
    "model__n_estimators": [50, 100, 150, 200, 250, 300, 400],
    "model__max_depth": [None, 5, 10, 15, 20, 25],}

pipe_rand = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))])

rand_search = RandomizedSearchCV(
    pipe_rand, param_distributions, n_iter=10, cv=5,
    scoring="neg_root_mean_squared_error", random_state=42, n_jobs=-1)

rand_search.fit(X_train, y_train)

cv_rand = -rand_search.best_score_
test_rand = np.sqrt(mean_squared_error(y_test, rand_search.best_estimator_.predict(X_test)))

print(f"RandomizedSearchCV RF -> CV RMSE: {cv_rand:.4f} | Test RMSE: {test_rand:.4f}")

RandomizedSearchCV RF -> CV RMSE: 0.5083 | Test RMSE: 0.5033


# Cell 10 — Assemble the master evaluation summary table

We build a consolidated pandas DataFrame showing Model Name, CV RMSE, Test RMSE, and Generalization Gap (|CV - Test|) for all models.

In [19]:
# Compile all results into a structured DataFrame.

results_data = [
    {"Model": "Linear Regression", "CV RMSE": cv_lr, "Test RMSE": test_lr},
    {"Model": "Gradient Boosting Regressor", "CV RMSE": cv_gb, "Test RMSE": test_gb},
    {"Model": "Random Forest (Default)", "CV RMSE": cv_rf, "Test RMSE": test_rf},
    {"Model": "Random Forest (GridSearchCV Best)", "CV RMSE": cv_grid, "Test RMSE": test_grid},
    {"Model": "Random Forest (RandomizedSearchCV Best)", "CV RMSE": cv_rand, "Test RMSE": test_rand},]

summary_table = pd.DataFrame(results_data)

# Compute absolute Generalization Gap between CV RMSE and Test RMSE

summary_table["Generalization Gap"] = np.abs(summary_table["CV RMSE"] - summary_table["Test RMSE"])

# Sort by Test RMSE in ascending order

summary_table = summary_table.sort_values(by="Test RMSE", ascending=True).reset_index(drop=True)
summary_table.index += 1
summary_table.index.name = "Rank"

summary_table.round(4)

,Model,CV RMSE,Test RMSE,Generalization Gap
Rank,,,,
1,Random Forest (RandomizedSearchCV Best),0.5083,0.5033,0.0051
2,Random Forest (GridSearchCV Best),0.5091,0.5039,0.0051
3,Random Forest (Default),0.5091,0.5039,0.0051
4,Gradient Boosting Regressor,0.5352,0.5417,0.0066
5,Linear Regression,0.7184,0.7441,0.0257


# Cell 11 — Final conclusions and analysis

We review the consolidated performance metrics and draw key takeaways regarding generalization and model selection.

In [20]:
# Report final observations on generalization and performance.
best_model_name = summary_table.iloc[0]["Model"]
best_test_rmse = summary_table.iloc[0]["Test RMSE"]

print("=== Exercise 4 Final Takeaways ===")
print(f"1. Top Performing Model on Test Data: {best_model_name}")
print(f"   - Lowest Test RMSE: {best_test_rmse:.4f}")
print("2. Generalization Consistency: All ensemble tree-based models exhibit a very small")
print("   Generalization Gap (~0.004 - 0.005), indicating excellent cross-validation alignment.")
print("3. Linear Regression shows significantly higher error on both CV and Test sets,")
print("   confirming non-linear ensemble models are superior for California Housing data.")

=== Exercise 4 Final Takeaways ===
1. Top Performing Model on Test Data: Random Forest (RandomizedSearchCV Best)
   - Lowest Test RMSE: 0.5033
2. Generalization Consistency: All ensemble tree-based models exhibit a very small
   Generalization Gap (~0.004 - 0.005), indicating excellent cross-validation alignment.
3. Linear Regression shows significantly higher error on both CV and Test sets,
   confirming non-linear ensemble models are superior for California Housing data.
